In [2]:
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

print(torch.__version__)

2.11.0


In [4]:
# Cargar df_feat ya procesado
df_tft = pd.read_csv('raw_data/df_feat_final.csv')
df_tft['ds'] = pd.to_datetime(df_tft['ds'])
df_tft = df_tft.sort_values('ds').reset_index(drop=True)

print(df_tft.shape)
print(df_tft.columns.tolist())

(98806, 21)
['y_scaled', 'hour', 'day_of_week', 'day_of_year', 'month', 'year', 'lag_1h', 'lag_24h', 'lag_168h', 'ma_24h', 'ma_168h', 'generation', 'consumption', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'is_holiday', 'is_weekend', 'ds', 'y']


In [5]:
df_tft = df_tft.sort_values('ds').reset_index(drop=True)
df_tft['time_idx'] = df_tft.index
df_tft['group'] = 'DE_LU'

print(df_tft[['ds', 'time_idx', 'group', 'y']].head())

                   ds  time_idx  group     y
0 2015-01-11 23:00:00         0  DE_LU  9.30
1 2015-01-12 00:00:00         1  DE_LU  2.87
2 2015-01-12 01:00:00         2  DE_LU  0.09
3 2015-01-12 02:00:00         3  DE_LU  0.26
4 2015-01-12 03:00:00         4  DE_LU  2.83


In [7]:
max_encoder_length = 168  # 1 semana de contexto
max_prediction_length = 24  # predecir 24 horas

# Split: test es el último 20%
training_cutoff = df_tft['time_idx'].max() - int(len(df_tft) * 0.2)

training = TimeSeriesDataSet(
    df_tft[df_tft['time_idx'] <= training_cutoff],
    time_idx='time_idx',
    target='y',
    group_ids=['group'],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_known_reals=['time_idx', 'hour', 'day_of_week', 'day_of_year', 'month', 'is_holiday', 'is_weekend'],
    time_varying_unknown_reals=['y', 'lag_1h', 'lag_24h', 'lag_168h', 'ma_24h', 'ma_168h',
                             'generation', 'consumption', 'temperature_c', 'humidity_percent',
                             'cloud_cover_percent', 'shortwave_radiation_wm2'],
    target_normalizer=GroupNormalizer(groups=['group']),
    add_relative_time_idx=True,
    add_target_scales=True,
)

validation = TimeSeriesDataSet.from_dataset(training, df_tft, predict=True, stop_randomization=True)

print(f'Train size: {len(training)}')
print(f'Val size: {len(validation)}')

Train size: 78854
Val size: 1


In [8]:
validation = TimeSeriesDataSet.from_dataset(training, df_tft[df_tft['time_idx'] > training_cutoff], stop_randomization=True)

print(f'Train size: {len(training)}')
print(f'Val size: {len(validation)}')

Train size: 78854
Val size: 19570


In [ ]:
tft_full = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=MAE(),
    log_interval=-1,
)


In [20]:
trainer = pl.Trainer(
    max_epochs=20,
    accelerator='cpu',
    gradient_clip_val=0.1,
    callbacks=[],
    logger=False,
    enable_progress_bar=True,
)

trainer.fit(tft_full, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/javierinocentezabala/code/grid-intelligence/notebooks/javier/checkpoints exists and is not empty.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    704 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  3.7 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 44.9 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 16.1 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 117 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 117 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 603                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=20` reached.


In [16]:
import pytorch_forecasting
import lightning
print(f'pytorch-forecasting: {pytorch_forecasting.__version__}')
print(f'lightning: {lightning.__version__}')

pytorch-forecasting: 1.7.0
lightning: 2.6.1


In [17]:
df_small = df_tft.head(5000).copy()
df_small['time_idx'] = range(len(df_small))

training_small = TimeSeriesDataSet(
    df_small[:4000],
    time_idx='time_idx',
    target='y',
    group_ids=['group'],
    max_encoder_length=24,
    max_prediction_length=24,
    time_varying_known_reals=['time_idx', 'hour', 'day_of_week', 'month', 'is_holiday', 'is_weekend'],
    time_varying_unknown_reals=['y', 'lag_1h', 'lag_24h', 'generation', 'consumption', 'temperature_c'],
    target_normalizer=GroupNormalizer(groups=['group']),
)

val_small = TimeSeriesDataSet.from_dataset(training_small, df_small[4000:], stop_randomization=True)

train_loader_small = training_small.to_dataloader(train=True, batch_size=64, num_workers=0)
val_loader_small = val_small.to_dataloader(train=False, batch_size=64, num_workers=0)

trainer_small = pl.Trainer(max_epochs=1, accelerator='cpu', callbacks=[], logger=False)
trainer_small.fit(tft, train_dataloaders=train_loader_small, val_dataloaders=val_loader_small)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ eval │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ eval │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ eval │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    704 │ eval │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  3.7 K │ eval │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 44.9 K │ eval │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 16.1 K │ eval │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ eval │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ eval │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ eval │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ eval │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ eval │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ eval │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ eval │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ eval │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ eval │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ eval │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ eval │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ eval │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ eval │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ eval │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 117 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 117 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 0                                                                                           
Modules in eval mode: 603                                                                                          
Total FLOPs: 0

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/rich/live.py
:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/py
torch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) 
and treespec.is_leaf()` instead.

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/py
torch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a 
bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to
improve performance.

IndexError: index 12 is out of bounds for dimension 1 with size 12

In [18]:
tft_small = TemporalFusionTransformer.from_dataset(
    training_small,
    learning_rate=0.001,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=MAE(),
    log_interval=-1,
)

trainer_small = pl.Trainer(max_epochs=1, accelerator='cpu', callbacks=[], logger=False)
trainer_small.fit(tft_small, train_dataloaders=train_loader_small, val_dataloaders=val_loader_small)

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless clou

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    192 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │      0 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  8.2 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  3.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     17 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 25.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 25.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 407                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/lightning/py
torch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a 
bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to
improve performance.

`Trainer.fit` stopped: `max_epochs=1` reached.
